In [ ]:
import Pkg
Pkg.activate(".")
Pkg.add([
    "Oceananigans",
    "NumericalEarth",
    "CUDA",
    "CairoMakie",
    "JLD2"
])

Pkg.precompile()

  Activating project at `c:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\simulation #1`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\simulation #1\Project.toml`
    Manifest No packages added to or removed from `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\simulation #1\Manifest.toml`


In [ ]:

using NumericalEarth
using Oceananigans
using Oceananigans.Units
using Dates
using Printf
using Statistics
using CUDA

using Oceananigans.Models: add_tracer!

In [ ]:

arch = GPU()


Nx = 360
Ny = 180
Nz = 50


depth = 5000meters

z = ExponentialDiscretization(Nz, -depth, 0; scale = depth/4, mutable = false)
underlying_grid = TripolarGrid(arch; size = (Nx, Ny, Nz), halo = (5, 5, 4), z)


bottom_height = regrid_bathymetry(underlying_grid;
                                  minimum_depth = 10,
                                  interpolation_passes = 10,
                                  major_basins = 2)

grid = ImmersedBoundaryGrid(underlying_grid, GridFittedBottom(bottom_height);
                            active_cells_map=true)

[ Info: Loading cached bathymetry from C:\Users\meghn\.julia\scratchspaces\904d977b-046a-4731-8b86-9235c0d1ef02\bathymetry_cache\bathymetry_360x180_0.0_360.0_-85.22387615721567_90.0_b841e80e.jld2


360×180×50 ImmersedBoundaryGrid{Float64, Periodic, RightCenterFolded, Bounded} on CUDAGPU with 5×5×4 halo:
├── immersed_boundary: GridFittedBottom(mean(z)=-2224.28, min(z)=-5000.0, max(z)=0.0)
├── underlying_grid: 360×180×50 OrthogonalSphericalShellGrid{Float64, Periodic, RightCenterFolded, Bounded} on CUDAGPU with 5×5×4 halo
├── centered at (λ, φ) = (250.0, 1.8005)
├── longitude: Periodic  extent 360.162 degrees         variably spaced with min(Δλ)=0.00753834, max(Δλ)=1.05373
├── latitude:  RightCenterFolded  extent 170.95 degrees variably spaced with min(Δφ)=0.0113516, max(Δφ)=0.949721
└── z:         Bounded  z ∈ [-5000.0, 0.0]              variably spaced with min(Δz)=7.76958, max(Δz)=391.59

In [ ]:

using Oceananigans.TurbulenceClosures: IsopycnalSkewSymmetricDiffusivity, AdvectiveFormulation

eddy_closure = IsopycnalSkewSymmetricDiffusivity(
    κ_skew = 1e3,
    κ_symmetric = 1e3,
    skew_flux_formulation = AdvectiveFormulation()
)

vertical_mixing = NumericalEarth.Oceans.default_ocean_closure()

CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
├── maximum_tracer_diffusivity: Inf
├── maximum_tke_diffusivity: Inf
├── maximum_viscosity: Inf
├── minimum_tke: 1.0e-9
├── negative_tke_time_scale: 60.0
├── minimum_convective_buoyancy_flux: 1.0e-11
├── tke_time_step: Nothing
├── mixing_length: TKEBasedVerticalDiffusivities.CATKEMixingLength
│   ├── Cˢ:   1.131
│   ├── Cᵇ:   0.01
│   ├── Cʰⁱu: 0.242
│   ├── Cʰⁱc: 0.098
│   ├── Cʰⁱe: 0.548
│   ├── Cˡᵒu: 0.361
│   ├── Cˡᵒc: 0.369
│   ├── Cˡᵒe: 7.863
│   ├── Cᵘⁿu: 0.37
│   ├── Cᵘⁿc: 0.572
│   ├── Cᵘⁿe: 1.447
│   ├── Cᶜu:  3.705
│   ├── Cᶜc:  4.793
│   ├── Cᶜe:  3.642
│   ├── Cᵉc:  0.112
│   ├── Cᵉe:  0.0
│   ├── Cˢᵖ:  0.505
│   ├── CRiᵟ: 1.02
│   └── CRi⁰: 0.254
└── turbulent_kinetic_energy_equation: TKEBasedVerticalDiffusivities.CATKEEquation
    ├── CʰⁱD: 0.579
    ├── CˡᵒD: 1.604
    ├── CᵘⁿD: 0.923
    ├── CᶜD:  3.254
    ├── CᵉD:  0.0
    ├── Cᵂu★: 3.179
    ├── CᵂwΔ: 0.383
    └── Cᵂϵ:  1.0

In [ ]:

free_surface = SplitExplicitFreeSurface(grid; substeps = 70)
momentum_advection = WENOVectorInvariant(order = 5)
tracer_advection   = WENO(order = 5)

# build the ocean model with grid, advection, free surface, closures, and the dye tracer
# dye is passive, so it gets moved + mixed but does not change the ocean physics
ocean = ocean_simulation(grid; momentum_advection, tracer_advection, free_surface,
                         closure = (eddy_closure, vertical_mixing),
                         tracers = (:T, :S, :dye))

# define the initial dye patch in the upper north atlantic
# this gives the dye a visible starting pattern instead of making it uniform everywhere
@inline function dye_initial_condition(x, y, z)
    in_lon_patch = (-70 <= x <= -30) || (290 <= x <= 330)
    in_lat_patch = 35 <= y <= 60
    in_depth_patch = z > -200

    return in_lon_patch && in_lat_patch && in_depth_patch ? 1.0 : 0.0
end

# print the model structure so we can check that dye is listed as a tracer
@info "We've built an ocean simulation with model:"
@show ocean.model

ocean.model = HydrostaticFreeSurfaceModel{CUDAGPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
├── grid: 360×180×50 ImmersedBoundaryGrid{Float64, Periodic, RightCenterFolded, Bounded} on CUDAGPU with 5×5×4 halo
├── timestepper: SplitRungeKuttaTimeStepper
├── tracers: (T, S, dye, e)
├── closure: Tuple with 2 closures:
│   ├── CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
│   └── IsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
├── free surface: SplitExplicitFreeSurface with gravitational acceleration 9.80665 m s⁻²
│   └── substepping: FixedSubstepNumber(50)
├── advection scheme: 
│   ├── momentum: WENOVectorInvariant{3, Float64}(vorticity_order=5, vertical_order=5)
│   ├── T: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   ├── S: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)

[ Info: We've built an ocean simulation with model:


HydrostaticFreeSurfaceModel{CUDAGPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
├── grid: 360×180×50 ImmersedBoundaryGrid{Float64, Periodic, RightCenterFolded, Bounded} on CUDAGPU with 5×5×4 halo
├── timestepper: SplitRungeKuttaTimeStepper
├── tracers: (T, S, dye, e)
├── closure: Tuple with 2 closures:
│   ├── CATKEVerticalDiffusivity{VerticallyImplicitTimeDiscretization}
│   └── IsopycnalSkewSymmetricDiffusivity(κ_skew=1000.0, κ_symmetric=1000.0)
├── buoyancy: SeawaterBuoyancy with g=9.80665 and BoussinesqEquationOfState{Float64} with ĝ = NegativeZDirection()
├── free surface: SplitExplicitFreeSurface with gravitational acceleration 9.80665 m s⁻²
│   └── substepping: FixedSubstepNumber(50)
├── advection scheme: 
│   ├── momentum: WENOVectorInvariant{3, Float64}(vorticity_order=5, vertical_order=5)
│   ├── T: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   ├── S: WENO{3, Float64, Oceananigans.Utils.BackendOptimizedDivision}(order=5)
│   ├── dye: 

In [ ]:

sea_ice = sea_ice_simulation(grid, ocean; advection=tracer_advection)

Simulation of SeaIceModel{CUDAGPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
├── Next time step: 5 minutes
├── run_wall_time: 0 seconds
├── run_wall_time / iteration: NaN days
├── stop_time: Inf days
├── stop_iteration: Inf
├── wall_time_limit: Inf
├── minimum_relative_step: 0.0
├── callbacks: OrderedDict with 3 entries:
│   ├── stop_time_exceeded => Callback of stop_time_exceeded on IterationInterval(1)
│   ├── stop_iteration_exceeded => Callback of stop_iteration_exceeded on IterationInterval(1)
│   └── wall_time_limit_exceeded => Callback of wall_time_limit_exceeded on IterationInterval(1)
└── output_writers: OrderedDict with no entries

In [ ]:
# ask for ecco credentials at runtime so they are not saved in the notebook
print("ECCO username: ")
ENV["ECCO_USERNAME"] = strip(readline())

print("ECCO WebDAV password: ")
ENV["ECCO_WEBDAV_PASSWORD"] = strip(readline())

println("ecco credentials saved in ENV for this session ✔️")
date = DateTime(1993, 1, 1)

ecco_variables = (:temperature, :salinity, :sea_ice_thickness, :sea_ice_concentration)
ecco_set = MetadataSet(ecco_variables; dataset = ECCO4Monthly(), date)

set!(ocean.model,   ecco_set)  

# initialize the dye field after ecco sets T + S
set!(ocean.model, dye = dye_initial_condition)
# check that the dye tracer exists fsr 
@show keys(ocean.model.tracers)

set!(sea_ice.model, ecco_set)  

keys(ocean.model.tracers) = (:T, :S, :dye, :e)


In [ ]:

land = JRA55PrescribedLand(arch)
atmosphere = JRA55PrescribedAtmosphere(arch)
ocean_surface = SurfaceRadiationProperties(albedo = LatitudeDependentAlbedo())
radiation = JRA55PrescribedRadiation(arch; ocean_surface)

640×320×1×2920 PrescribedRadiation on LatitudeLongitudeGrid:
├── times: 2920-element StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}
├── stefan_boltzmann_constant: 5.67037e-8
└── surface_properties: (:ocean, :sea_ice)

In [ ]:

coupled_model = OceanSeaIceModel(ocean, sea_ice; atmosphere, land, radiation)

#model runs for 90 days with a timestep of 20 
simulation = Simulation(coupled_model; Δt=20minutes, stop_time=90days)

┌ Warning: 1440×720×1×365 PrescribedLand tracks time as Float32 but the EarthSystemModel clock uses Float64; coercing the component clock to keep components synchronized.
└ @ NumericalEarth.EarthSystemModels C:\Users\meghn\.julia\packages\NumericalEarth\eKhWQ\src\EarthSystemModels\components.jl:136


Simulation of EarthSystemModel{GPU}(time = 0 seconds, iteration = 0)
├── Next time step: 20 minutes
├── run_wall_time: 0 seconds
├── run_wall_time / iteration: NaN days
├── stop_time: 90 days
├── stop_iteration: Inf
├── wall_time_limit: Inf
├── minimum_relative_step: 0.0
├── callbacks: OrderedDict with 4 entries:
│   ├── stop_time_exceeded => Callback of stop_time_exceeded on IterationInterval(1)
│   ├── stop_iteration_exceeded => Callback of stop_iteration_exceeded on IterationInterval(1)
│   ├── wall_time_limit_exceeded => Callback of wall_time_limit_exceeded on IterationInterval(1)
│   └── nan_checker => Callback of NaNChecker for T_ocean on IterationInterval(100)
└── output_writers: OrderedDict with no entries

In [ ]:

wall_time = Ref(time_ns())

function progress(sim)
 
    ocean = sim.model.ocean

    u, v, w = ocean.model.velocities

    T = ocean.model.tracers.T
    e = ocean.model.tracers.e

    # grab the dye tracer so it gets saved with the ocean output
    dye = ocean.model.tracers.dye

    Tmin, Tmax, Tavg = minimum(T), maximum(T), mean(view(T, :, :, ocean.model.grid.Nz))
    emax = maximum(e)

    # compute max speed components, so we can see if the model is behaving weirdly
    umax = (maximum(abs, u), maximum(abs, v), maximum(abs, w))

    step_time = 1e-9 * (time_ns() - wall_time[])

    msg1 = @sprintf("time: %s, iter: %d", prettytime(sim), iteration(sim))
    msg2 = @sprintf(", max|uo|: (%.1e, %.1e, %.1e) m s⁻¹", umax...)
    msg3 = @sprintf(", extrema(To): (%.1f, %.1f) ᵒC, mean(To(z=0)): %.1f ᵒC", Tmin, Tmax, Tavg)
    msg4 = @sprintf(", max(e): %.2f m² s⁻²", emax)
    msg5 = @sprintf(", wall time: %s \n", prettytime(step_time))

    @info msg1 * msg2 * msg3 * msg4 * msg5

    wall_time[] = time_ns()

     return nothing
end

progress (generic function with 1 method)

In [ ]:

add_callback!(simulation, progress, TimeInterval(5days))

In [ ]:
# this collects the ocean tracers + velocities into one output group
# tracers are T, S, e, and dye,  while velocities are u, v, and w

ocean_outputs = merge(ocean.model.tracers, ocean.model.velocities)

free_surface = ocean.model.free_surface.displacement

sea_ice_outputs = merge((h = sea_ice.model.ice_thickness,
                         ℵ = sea_ice.model.ice_concentration,
                         T = sea_ice.model.ice_thermodynamics.top_surface_temperature),
                         sea_ice.model.velocities)

ocean.output_writers[:surface] = JLD2Writer(ocean.model, ocean_outputs;
                                            schedule = TimeInterval(1days),
                                            filename = "ocean_one_degree_surface_fields",
                                            indices = (:, :, grid.Nz),
                                            overwrite_existing = true)


ocean.output_writers[:free_surface] = JLD2Writer(ocean.model, (; η = free_surface);
                                                 schedule = TimeInterval(1days),
                                                 filename = "ocean_one_degree_free_surface",
                                                 overwrite_existing = true)


sea_ice.output_writers[:surface] = JLD2Writer(sea_ice.model, sea_ice_outputs;
                                              schedule = TimeInterval(1days),
                                              filename = "sea_ice_one_degree_surface_fields",
                                              overwrite_existing = true)

JLD2Writer scheduled on TimeInterval(1 day):
├── filepath: sea_ice_one_degree_surface_fields.jld2
├── 5 outputs: (h, ℵ, T, u, v)
├── array_type: Array{Float32}
├── including: [:grid]
├── file_splitting: NoFileSplitting
└── file size: 0 bytes (file not yet created)

In [ ]:

run!(simulation)

[ Info: Initializing simulation...
[ Info: time: 0 seconds, iter: 0, max|uo|: (0.0e+00, 0.0e+00, 0.0e+00) m s⁻¹, extrema(To): (-1.9, 31.3) ᵒC, mean(To(z=0)): 15.9 ᵒC, max(e): 0.00 m² s⁻², wall time: 1.169 minutes 
[ Info:     ... simulation initialization complete (24.187 seconds)
[ Info: Executing initial time step...
┌ Warning: error ArgumentError("a group or dataset named grid is already present within this group") thrown when trying to serialize the grid at serialized/grid
└ @ Oceananigans.OutputWriters C:\Users\meghn\.julia\packages\Oceananigans\NCFoc\src\OutputWriters\jld2_writer.jl:234
[ Info:     ... initial time step complete (8.745 minutes).
[ Info: time: 5 days, iter: 360, max|uo|: (1.5e+00, 1.7e+00, 1.4e-02) m s⁻¹, extrema(To): (-1.9, 31.6) ᵒC, mean(To(z=0)): 16.2 ᵒC, max(e): 0.02 m² s⁻², wall time: 21.894 minutes 
[ Info: time: 10 days, iter: 720, max|uo|: (1.6e+00, 2.1e+00, 1.4e-02) m s⁻¹, extrema(To): (-1.9, 32.2) ᵒC, mean(To(z=0)): 16.3 ᵒC, max(e): 0.01 m² s⁻², wall tim

In [ ]:

using CairoMakie

uo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "u"; backend = OnDisk())
vo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "v"; backend = OnDisk())
To = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "T"; backend = OnDisk())
eo = FieldTimeSeries("ocean_one_degree_surface_fields.jld2",  "e"; backend = OnDisk())
ηo = FieldTimeSeries("ocean_one_degree_free_surface.jld2",    "η"; backend = OnDisk())

# load dye tracer
dye = FieldTimeSeries("ocean_one_degree_surface_fields.jld2", "dye"; backend = OnDisk())

360×180×1×91 FieldTimeSeries{OnDisk} located at (Center, Center, Center) of dye at ocean_one_degree_surface_fields.jld2
├── grid: 360×180×50 ImmersedBoundaryGrid{Float64, Periodic, RightCenterFolded, Bounded} on CPU with 5×5×4 halo
├── indices: (:, :, 50:50)
├── time_indexing: Linear()
├── backend: OnDisk
├── path: ocean_one_degree_surface_fields.jld2
└── name: dye

In [ ]:

ui = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "u"; backend = OnDisk())
vi = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "v"; backend = OnDisk())
hi = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "h"; backend = OnDisk())
ℵi = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "ℵ"; backend = OnDisk())
Ti = FieldTimeSeries("sea_ice_one_degree_surface_fields.jld2", "T"; backend = OnDisk())

times = uo.times
Nt = length(times)
n = Observable(Nt)

Observable(91)


In [ ]:

land = interior(To.grid.immersed_boundary.bottom_height) .≥ 0

Toₙ = @lift begin
    Tₙ = interior(To[$n])
    Tₙ[land] .= NaN
    view(Tₙ, :, :, 1)
end


eoₙ = @lift begin
    eₙ = interior(eo[$n])
    eₙ[land] .= NaN
    view(eₙ, :, :, 1)
end

ηoₙ = @lift begin
    ηₙ = interior(ηo[$n])
    ηₙ[land] .= NaN
    view(ηₙ, :, :, 1)
end

heₙ = @lift begin
    hₙ = interior(hi[$n])
    ℵₙ = interior(ℵi[$n])
    hₙ[land] .= NaN
    view(hₙ, :, :, 1) .* view(ℵₙ, :, :, 1)
end

# update the dye field whenever the animation frame changes
dyeₙ = @lift begin
    d = interior(dye[$n])
    d[land] .= NaN
    view(d, :, :, 1)
end

uoₙ = Field{Face, Center, Nothing}(uo.grid)
voₙ = Field{Center, Face, Nothing}(vo.grid)
uiₙ = Field{Face, Center, Nothing}(ui.grid)
viₙ = Field{Center, Face, Nothing}(vi.grid)
so = Field(sqrt(uoₙ^2 + voₙ^2))
si = Field(sqrt(uiₙ^2 + viₙ^2))
soₙ = @lift begin
    parent(uoₙ) .= parent(uo[$n])
    parent(voₙ) .= parent(vo[$n])
    compute!(so)
    soₙ = interior(so)
    soₙ[land] .= NaN
    view(soₙ, :, :, 1)
end

siₙ = @lift begin
    parent(uiₙ) .= parent(ui[$n])
    parent(viₙ) .= parent(vi[$n])
    compute!(si)
    siₙ = interior(si)
    hₙ = interior(hi[$n])
    ℵₙ = interior(ℵi[$n])
    he = hₙ .* ℵₙ
    siₙ[he .< 1e-7] .= 0
    siₙ[land] .= NaN
    view(siₙ, :, :, 1)
end

Observable([NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN; NaN NaN … NaN NaN])


In [ ]:

fig = Figure(size=(1200, 1300))

title = @lift string("Global 1ᵒ ocean simulation after ", prettytime(times[$n] - times[1]))

axso = Axis(fig[1, 1])
axηo = Axis(fig[1, 3])
axTo = Axis(fig[2, 1])
axeo = Axis(fig[2, 3])
axsi = Axis(fig[3, 1])
axhi = Axis(fig[3, 3])
# this adds one more panel for the passive dye tracer
axdye = Axis(fig[4, 1])


hm = heatmap!(axso, soₙ, colorrange = (0, 0.5), colormap = :deep,  nan_color=:lightgray)
Colorbar(fig[1, 2], hm, label = "Ocean Surface speed (m s⁻¹)")


hm = heatmap!(axηo, ηoₙ, colorrange = (-1.2, 1.2), colormap = :balance, nan_color=:lightgray)
Colorbar(fig[1, 4], hm, label = "Sea Surface Height (m)")


hm = heatmap!(axTo, Toₙ, colorrange = (-1, 32), colormap = :magma, nan_color=:lightgray)
Colorbar(fig[2, 2], hm, label = "Surface Temperature (ᵒC)")


hm = heatmap!(axeo, eoₙ, colorrange = (0, 1e-3), colormap = :solar, nan_color=:lightgray)
Colorbar(fig[2, 4], hm, label = "Turbulent Kinetic Energy (m² s⁻²)")


hm = heatmap!(axsi, siₙ, colorrange = (0, 0.5), colormap = :greys, nan_color=:lightgray)
Colorbar(fig[3, 2], hm, label = "Sea ice speed (m s⁻¹)")


hm = heatmap!(axhi, heₙ, colorrange =  (0, 4),  colormap = :blues, nan_color=:lightgray)
Colorbar(fig[3, 4], hm, label = "Effective ice thickness (m)")

# passive dye tracer
hm = heatmap!(axdye, dyeₙ, colorrange = (0, 1), colormap = :viridis, nan_color=:lightgray)
Colorbar(fig[4, 2], hm, label = "Passive dye concentration")

for ax in (axso, axsi, axTo, axhi, axeo, axdye)
    hidedecorations!(ax)
end

Label(fig[0, :], title)


save("global_snapshot.png", fig)

In [ ]:

CairoMakie.record(fig, "one_degree_global_ocean_surface.mp4", 1:Nt, framerate = 8) do nn
    n[] = nn
end

"one_degree_global_ocean_surface.mp4"